# 💻 GUÍA PRÁCTICA AVANZADA: ARQUITECTURA MODEL CONTEXT PROTOCOL (MCP)
## Visualizando Paquetes JSON-RPC, Servidores Web SSE e Independencia de Modelos (GPT-4o / Gemini / Nemotron)

En esta versión extendida del curso interactivo llevaremos el concepto de MCP a un nivel técnico y didáctico sumamente alto. Vamos a implementar:
1. **Visualización de Paquetes JSON-RPC (Tuberías en Vivo)**: Cada vez que nos comuniquemos, dibujaremos tarjetas HTML de diseño premium en la consola que muestren las peticiones y respuestas JSON crudas del estándar MCP.
2. **Soporte de Transporte SSE (Server-Sent Events)**: Podrás elegir entre conectar el servidor local de forma oculta en consola (`stdio`) o levantar un servidor Web SSE API real en `http://localhost:8000/sse`.
3. **Intercambiabilidad de Cerebro de IA**: Podrás alternar con un menú interactivo en tiempo de ejecución entre diferentes modelos líderes (Nvidia Nemotron, GPT-4o Mini o Gemini) utilizando tu nueva clave de API unificada, demostrando que el código del Servidor de base de datos nunca cambia.

---

## 📦 Paso 1: Instalación de Dependencias (Específico para Databricks)
Para asegurar que este notebook sea completamente autogestionado y reproducible dentro de tu clúster de Databricks, instalamos las librerías necesarias usando comandos mágicos de clúster y reiniciamos el kernel de Python para cargarlas de inmediato.

In [ ]:
# Comando mágico de Databricks para instalar todas las dependencias del proyecto en el clúster
%pip install mcp openai pandas scikit-learn numpy sqlalchemy pymysql python-dotenv

In [ ]:
# Reiniciar el entorno de ejecución de Python en el clúster para registrar los nuevos paquetes instalados
# Puedes usar %restart_python en versiones modernas de Databricks, o dbutils.library.restartPython() como alternativa clásica.
%restart_python
# dbutils.library.restartPython()

---

## 🛠️ Paso 2: Importar Librerías y Herramientas Visuales
Importamos los componentes estándar de MCP y OpenAI, junto con las utilidades de visualización web de Jupyter (`HTML`, `display`).

In [ ]:
import os
import asyncio
import json
from dotenv import load_dotenv
from openai import AsyncOpenAI
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from mcp.client.sse import sse_client
from IPython.display import display, HTML

---

## 🔑 Paso 3: Carga Segura y Verificación de Credenciales (.env)
Cargamos y evaluamos que tu variable de API Key unificada esté correctamente configurada para evitar fallos de ejecución.

In [ ]:
# Cargar variables del .env
load_dotenv(override=True)

openrouter_key = os.getenv("OPENROUTER_API_KEY")

if not openrouter_key or openrouter_key.strip() == "" or "tu_api_key" in openrouter_key:
    raise ValueError(
        "❌ [ERROR DE CREDENCIALES] La variable 'OPENROUTER_API_KEY' no está configurada o es inválida.\n"
        "Por favor, abre tu archivo local '.env' y añade una clave de API válida para continuar."
    )
else:
    print("✅ Credenciales y variables de entorno cargadas con éxito. Listo para la inferencia.")

---

## 🌐 Paso 4: Selección de Transporte (¿Stdio o SSE?)
Configuramos el tipo de transporte que usará el cliente para comunicarse con el servidor:
- `stdio`: El Notebook iniciará el servidor en segundo plano de manera oculta y local (ideal para Databricks).
- `sse`: Se conecta a un servidor web MCP ya activo en `http://localhost:8000/sse`.

*(Si eliges SSE, recuerda ejecutar primero `uv run python mcp_server.py sse` en una terminal externa de tu máquina)*.

In [ ]:
print("=======================================================================")
print(" SELECCIÓN DE TRANSPORTE MCP:")
print("-----------------------------------------------------------------------")
print(" [1] -> stdio (Servidor integrado por consola - POR DEFECTO Y RECOMENDADO)")
print(" [2] -> sse   (Servidor Web local por red http://localhost:8000/sse)")
print("=======================================================================\n")

opcion_transporte = input("🔌 Selecciona el transporte (1 o 2, ENTER para predeterminado [1]): ").strip()
transporte_mcp = "sse" if opcion_transporte == "2" else "stdio"

# Configurar los parámetros si usamos stdio
server_params = StdioServerParameters(
    command="uv",
    args=["run", "python", "mcp_server.py"],
    env=os.environ.copy()
)

print(f"\n✅ Transporte MCP configurado en modo: '{transporte_mcp.upper()}'")

---

## 🎨 Paso 5: Definición del Sniffer Visual de Paquetes JSON-RPC
Creamos una función que intercepta los flujos y dibuja tarjetas interactivas de diseño premium con efecto neón/glassmorphism en tu Notebook para mostrar exactamente qué JSON viaja por el estándar MCP.

In [ ]:
def mostrar_log_visual_json_rpc(direccion, metodo, payload):
    """Dibuja una tarjeta con diseño premium para visualizar el flujo JSON-RPC de MCP."""
    if direccion == "CLIENTE_ENVIANDO":
        titulo = "📤 CLIENTE MCP -> SOLICITANDO ACCIÓN (JSON-RPC OUTBOX)"
        borde_color = "#00c3ff"
        bg_color = "rgba(0, 195, 255, 0.05)"
    elif direccion == "SERVIDOR_RESPONDIENDO":
        titulo = "📥 SERVIDOR MCP -> RETORNANDO RESPUESTA (JSON-RPC INBOX)"
        borde_color = "#00ff88"
        bg_color = "rgba(0, 255, 136, 0.05)"
    else:
        titulo = "🧠 INTELIGENCIA ARTIFICIAL -> DECISIÓN DE INVOCAR HERRAMIENTA"
        borde_color = "#ffaa00"
        bg_color = "rgba(255, 170, 0, 0.05)"
        
    json_bonito = json.dumps(payload, indent=2, ensure_ascii=False)
    
    html_code = f"""
    <div style="
        border: 2px solid {borde_color};
        background-color: {bg_color};
        color: #e2e8f0;
        padding: 15px;
        border-radius: 8px;
        margin: 15px 0;
        font-family: 'Consolas', monospace;
        box-shadow: 0 4px 15px rgba(0, 0, 0, 0.3);
    ">
        <div style="font-weight: bold; color: {borde_color}; margin-bottom: 8px; font-size: 1.1em;">
            {titulo}
        </div>
        <div style="font-size: 0.9em; margin-bottom: 5px; color: #94a3b8;">
            <strong>Método/Acción:</strong> <code>{metodo}</code>
        </div>
        <pre style="
            background-color: #0f172a;
            padding: 10px;
            border-radius: 5px;
            overflow-x: auto;
            border: 1px solid #1e293b;
            color: #38bdf8;
            margin: 0;
        ">{json_bonito}</pre>
    </div>
    """
    display(HTML(html_code))

---

## 💬 Paso 6: Configuración Interactiva de la Consulta (Prompt)
Se abrirá una casilla interactiva. Presiona Enter directamente para usar el prompt predeterminado o escribe tu propio requerimiento.

In [ ]:
prompt_predeterminado = (
    "Por favor, extrae 100 transacciones de la base de datos MySQL "
    "utilizando tus herramientas locales y luego procesa los datos "
    "para dejarlos listos para entrenar un modelo de ML."
)

print("=======================================================================")
print(" SUGERENCIAS PARA EL MODELO DE IA:")
print("-----------------------------------------------------------------------")
print(" [Por defecto] -> (Solo presiona Enter en la casilla)")
print(f"   '{prompt_predeterminado}'\n")
print(" [Sugerencia 1] -> 'Extrae solo 10 transacciones de MySQL y no hagas nada más'")
print("=======================================================================\n")

user_input = input("✍️ Escribe tu instrucción (o presiona ENTER para usar la predeterminada): ").strip()
prompt_activo = user_input if user_input else prompt_predeterminado
print(f"\n✅ Prompt configurado: '{prompt_activo}'")

---

## 🤖 Paso 7: Selección Dinámica de Cerebros de IA (Multi-Model Swapping)
Elige con qué cerebro de IA procesar la tarea. Al cambiarlo, verás que el comportamiento del Agente cambia en su nivel de abstracción, pero la base de datos y el servidor MCP se mantienen idénticos.

In [ ]:
print("=======================================================================")
print(" SELECCIÓN DE MODELO DE INTELIGENCIA ARTIFICIAL:")
print("-----------------------------------------------------------------------")
print(" [1] -> nvidia/nemotron-3-super-120b-a12b:free (Nemotron - Gratis e Inteligente)")
print(" [2] -> google/gemini-2.5-flash (Gemini 2.5 Flash - Excelente orquestación)")
print(" [3] -> openai/gpt-4o-mini (GPT-4o Mini - Altamente resolutivo para Tools)")
print("=======================================================================\n")

opcion_modelo = input("🧠 Selecciona el modelo (1, 2 o 3, ENTER para predeterminado [1]): ").strip()

if opcion_modelo == "2":
    modelo_activo = "google/gemini-2.5-flash"
elif opcion_modelo == "3":
    modelo_activo = "openai/gpt-4o-mini"
else:
    modelo_activo = "nvidia/nemotron-3-super-120b-a12b:free"
    
print(f"\n✅ Inferencia configurada sobre el modelo: '{modelo_activo}'")

---

## 🔌 Paso 8: Inicialización del Cliente OpenAI / OpenRouter
Instanciamos nuestro cliente de OpenRouter para conectarnos con el modelo y transporte seleccionados.

In [ ]:
llm_client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)
print("✅ Cliente API inicializado y listo.")

---

## 🚀 Paso 9: Conexión y Orquestación del Pipeline MCP con Logs JSON-RPC
Esta celda ejecuta la lógica unificada del Cliente MCP. 
1. Se conecta mediante el transporte configurado (Stdio o SSE).
2. Ejecuta el handshake y dibuja los paquetes JSON-RPC enviados y recibidos.
3. Genera la consulta al LLM.
4. Intercepta los Tool Calls de la IA y los ejecuta localmente, graficando las llamadas de backend con diseño premium.

In [ ]:
async def correr_pipeline_mcp_avanzado(prompt, modelo):
    # Definición de conexión dinámica basada en tu transporte elegido en el Paso 4
    if transporte_mcp == "sse":
        cliente_mcp = sse_client("http://localhost:8000/sse")
    else:
        cliente_mcp = stdio_client(server_params)
        
    print(f"🔌 [MCP] Conectando vía transporte {transporte_mcp.upper()}...")
    
    async with cliente_mcp as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            # A. LOG VISUAL JSON-RPC: Solicitando herramientas al servidor
            mostrar_log_visual_json_rpc(
                direccion="CLIENTE_ENVIANDO",
                metodo="tools/list",
                payload={"jsonrpc": "2.0", "method": "tools/list", "id": 1}
            )
            
            # Handshake real
            tools_response = await session.list_tools()
            
            # B. LOG VISUAL JSON-RPC: Imprimiendo el catálogo retornado por el servidor
            esquema_tools_servidor = {
                "jsonrpc": "2.0",
                "result": {
                    "tools": [
                        {"name": t.name, "description": t.description, "inputSchema": t.inputSchema}
                        for t in tools_response.tools
                    ]
                },
                "id": 1
            }
            mostrar_log_visual_json_rpc(
                direccion="SERVIDOR_RESPONDIENDO",
                metodo="tools/list (RESPONSE)",
                payload=esquema_tools_servidor
            )
            
            # C. Traducir catálogo MCP a formato OpenAI
            openrouter_tools = []
            for tool in tools_response.tools:
                openrouter_tools.append({
                    "type": "function",
                    "function": {
                        "name": tool.name,
                        "description": tool.description,
                        "parameters": tool.inputSchema
                    }
                })
            
            # D. Armamos el mensaje para el LLM
            messages = [
                {
                    "role": "system",
                    "content": "Eres un agente científico de datos altamente técnico. Tu tarea es extraer la data de MySQL y transformarla usando tus herramientas locales. Responde de forma muy concisa."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ]
            
            # E. Inferencia con el modelo activo
            print(f"🧠 [LLM] Consultando decisiones al modelo '{modelo}'...")
            response = await llm_client.chat.completions.create(
                model=modelo,
                messages=messages,
                tools=openrouter_tools
            )
            
            mensaje_modelo = response.choices[0].message
            
            # F. Procesar Tool Calls de manera visual
            if mensaje_modelo.tool_calls:
                for id_call, tool_call in enumerate(mensaje_modelo.tool_calls):
                    nombre_funcion = tool_call.function.name
                    argumentos = json.loads(tool_call.function.arguments)
                    
                    # LOG VISUAL: La IA ha dictado usar una herramienta
                    mostrar_log_visual_json_rpc(
                        direccion="IA_DECIDIENDO",
                        metodo=f"tool_call -> {nombre_funcion}",
                        payload={
                            "tool_call_id": tool_call.id,
                            "model": modelo,
                            "action_requested": {
                                "name": nombre_funcion,
                                "arguments": argumentos
                            }
                        }
                    )
                    
                    # LOG VISUAL: Enviando petición de ejecución local al servidor MCP
                    mcp_request_payload = {
                        "jsonrpc": "2.0",
                        "method": "tools/call",
                        "params": {
                            "name": nombre_funcion,
                            "arguments": argumentos
                        },
                        "id": 10 + id_call
                    }
                    mostrar_log_visual_json_rpc(
                        direccion="CLIENTE_ENVIANDO",
                        metodo="tools/call (REQUEST)",
                        payload=mcp_request_payload
                    )
                    
                    # EJECUCIÓN REAL en mcp_server.py
                    resultado_local = await session.call_tool(nombre_funcion, argumentos)
                    
                    # LOG VISUAL: Recibiendo respuesta con los datos cargados desde el backend local
                    mcp_response_payload = {
                        "jsonrpc": "2.0",
                        "result": {
                            "content": [
                                {"type": "text", "text": resultado_local.content[0].text}
                            ]
                        },
                        "id": 10 + id_call
                    }
                    mostrar_log_visual_json_rpc(
                        direccion="SERVIDOR_RESPONDIENDO",
                        metodo="tools/call (RESPONSE)",
                        payload=mcp_response_payload
                    )
                print("🎉 Pipeline MCP orquestado con éxito.")
            else:
                print("\n💬 Respuesta del modelo (sin llamadas a herramientas):")
                print(mensaje_modelo.content)

print("Orquestador de visualización avanzada listo.")

---

## 🚀 Paso 10: Iniciar Inferencia y Ejecución en Vivo
Corremos la orquestación. Si elegiste transporte stdio, todo iniciará automáticamente. Si elegiste SSE, asegúrate de tener corriendo tu comando de servidor en tu terminal externa.

In [ ]:
await correr_pipeline_mcp_avanzado(prompt_activo, modelo_activo)